In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep
from pynq import Overlay, allocate

# ===== User parameters =====
BITFILE = 'base_add.bit'
PL_CLK_HZ = 125_000_000
ADC_HALF_PERIOD = 3125      # 125 MHz / (2 * 3125) = 20 kS/s
FS_HZ = PL_CLK_HZ / (2 * ADC_HALF_PERIOD)
SAMPLE_COUNT = 65536       # about 3.2768 s at 20 kS/s
CAPTURE_MODE = 1           # 1 = real AD9226, 2 = RTL fake stream
CHANNEL = 'ch0'            # 'ch0' or 'ch1'

# ===== AFSK/SMS parameters =====
SPACE_HZ = 1200.0          # bit 0
MARK_HZ = 2200.0           # bit 1
BIT_RATE = 100.0
BIT_SAMPLES = int(round(FS_HZ / BIT_RATE))
CENTER_FRACTION = 0.60     # use middle 6 ms of each 10 ms bit
OFFSET_STRIDE = 4          # 4 samples = 0.2 ms at 20 kS/s; smaller is slower
MIN_PREAMBLE_BITS = 24     # 3 bytes of 0101 is enough to declare preamble seen
PREAMBLE_BYTES = [0xAA, 0xAA, 0xAA, 0xAA]
FLAG_BYTE = 0x7E
LOCAL_ADDR = None          # set to 0..7 to filter, None accepts all plus 0xFF

# ===== Register offsets =====
CTRL = 0x00
STATUS = 0x04
SAMPLE_COUNT_REG = 0x08
ADC_HALF = 0x0C
SAMPLE_DELAY = 0x10
DECIMATION = 0x14
CHANNEL_MASK = 0x18
CAPTURE_MODE_REG = 0x1C
TRIGGER_MODE = 0x20
PRE_DELAY = 0x24
BUFFER_SELECT = 0x28
ERROR_FLAGS = 0x3C
AXIS_SENT_COUNT = 0x54
TLAST_COUNT = 0x5C
DROPPED_SAMPLE_COUNT = 0x64
S2MM_DMASR = 0x34
SENTINEL = np.uint32(0xDEADBEEF)

def find_overlay_attr(overlay, text):
    matches = [name for name in overlay.ip_dict.keys() if text in name.lower()]
    if not matches:
        raise RuntimeError(f"Cannot find IP containing '{text}'. IPs: {list(overlay.ip_dict.keys())}")
    name = matches[0]
    if '/' in name:
        raise RuntimeError(f"IP '{name}' is hierarchical; bind it manually from overlay.ip_dict.")
    return name, getattr(overlay, name)

overlay = Overlay(BITFILE)
ctrl_name, ctrl = find_overlay_attr(overlay, 'adc_capture')
dma_name, dma = find_overlay_attr(overlay, 'dma')

def configure_capture(sample_count=SAMPLE_COUNT, adc_half_period=ADC_HALF_PERIOD, capture_mode=CAPTURE_MODE):
    ctrl.write(CTRL, 0x04)
    ctrl.write(CTRL, 0x00)
    ctrl.write(ERROR_FLAGS, 0xFFFFFFFF)
    ctrl.write(SAMPLE_COUNT_REG, int(sample_count))
    ctrl.write(ADC_HALF, int(adc_half_period))
    ctrl.write(SAMPLE_DELAY, 1)
    ctrl.write(DECIMATION, 1)
    ctrl.write(CHANNEL_MASK, 0b11)
    ctrl.write(CAPTURE_MODE_REG, int(capture_mode))
    ctrl.write(TRIGGER_MODE, 0)
    ctrl.write(PRE_DELAY, 0)
    ctrl.write(BUFFER_SELECT, 0)

def capture_once(sample_count=SAMPLE_COUNT, adc_half_period=ADC_HALF_PERIOD, capture_mode=CAPTURE_MODE):
    sample_count = int(min(max(sample_count, 1), 65536))
    buf = allocate(shape=(sample_count,), dtype=np.uint32)
    buf[:] = SENTINEL
    buf.flush()
    dma.recvchannel.transfer(buf)
    configure_capture(sample_count, adc_half_period, capture_mode)
    ctrl.write(CTRL, 0x01)
    ctrl.write(CTRL, 0x03)
    ctrl.write(CTRL, 0x01)
    dma.recvchannel.wait()
    buf.invalidate()
    raw = np.array(buf, dtype=np.uint32)
    ch0 = raw & np.uint32(0x0FFF)
    ch1 = (raw >> np.uint32(16)) & np.uint32(0x0FFF)
    if np.any(raw == SENTINEL):
        raise RuntimeError('DMA buffer still contains sentinel values.')
    if ctrl.read(AXIS_SENT_COUNT) != sample_count:
        raise RuntimeError('AXIS_SENT_COUNT mismatch.')
    if ctrl.read(TLAST_COUNT) != 1:
        raise RuntimeError('TLAST_COUNT is not 1.')
    if ctrl.read(DROPPED_SAMPLE_COUNT) != 0:
        raise RuntimeError('Dropped samples detected.')
    return raw, ch0.astype(np.float64), ch1.astype(np.float64)

def tone_power(samples, freq_hz):
    x = samples.astype(np.float64)
    x = x - np.mean(x)
    n = np.arange(len(x), dtype=np.float64)
    phase = 2.0 * np.pi * freq_hz * n / FS_HZ
    i = np.sum(x * np.cos(phase))
    q = np.sum(x * np.sin(phase))
    return i * i + q * q

def decide_bit(bit_samples):
    margin = (1.0 - CENTER_FRACTION) * 0.5
    start = int(round(len(bit_samples) * margin))
    stop = int(round(len(bit_samples) * (1.0 - margin)))
    center = bit_samples[start:stop]
    p0 = tone_power(center, SPACE_HZ)
    p1 = tone_power(center, MARK_HZ)
    bit = 1 if p1 > p0 else 0
    confidence = abs(p1 - p0) / max(p0 + p1, 1.0)
    return bit, confidence, p0, p1

def crc8_0x07(data):
    crc = 0
    for byte in data:
        crc ^= byte & 0xFF
        for _ in range(8):
            crc = ((crc << 1) ^ 0x07) & 0xFF if (crc & 0x80) else (crc << 1) & 0xFF
    return crc

def bits_to_bytes_lsb(bits, bit_phase):
    out = []
    for start in range(bit_phase, len(bits) - 7, 8):
        value = 0
        for i in range(8):
            value |= (bits[start + i] & 1) << i
        out.append(value)
    return out

def find_frame(byte_values):
    sync = PREAMBLE_BYTES + [FLAG_BYTE]
    for i in range(0, len(byte_values) - len(sync) - 3):
        if byte_values[i:i + len(sync)] != sync:
            continue
        base = i + len(sync)
        addr = byte_values[base]
        length = byte_values[base + 1]
        end = base + 2 + length + 1
        if end > len(byte_values):
            continue
        payload = byte_values[base + 2:base + 2 + length]
        rx_crc = byte_values[base + 2 + length]
        calc_crc = crc8_0x07([addr, length] + payload)
        addr_ok = (LOCAL_ADDR is None) or (addr == LOCAL_ADDR) or (addr == 0xFF)
        return {
            'byte_index': i,
            'addr': addr,
            'length': length,
            'payload': payload,
            'rx_crc': rx_crc,
            'calc_crc': calc_crc,
            'crc_ok': rx_crc == calc_crc,
            'addr_ok': addr_ok,
        }
    return None

def demod_bits(samples, offset):
    samples = samples.astype(np.float64)
    samples = samples - np.mean(samples)
    margin = (1.0 - CENTER_FRACTION) * 0.5
    center_start = int(round(BIT_SAMPLES * margin))
    center_stop = int(round(BIT_SAMPLES * (1.0 - margin)))
    win_n = center_stop - center_start
    n = np.arange(win_n, dtype=np.float64)
    c0 = np.cos(2.0 * np.pi * SPACE_HZ * n / FS_HZ)
    s0 = np.sin(2.0 * np.pi * SPACE_HZ * n / FS_HZ)
    c1 = np.cos(2.0 * np.pi * MARK_HZ * n / FS_HZ)
    s1 = np.sin(2.0 * np.pi * MARK_HZ * n / FS_HZ)
    bits = []
    confidences = []
    count = (len(samples) - offset) // BIT_SAMPLES
    for k in range(count):
        start = offset + k * BIT_SAMPLES + center_start
        stop = offset + k * BIT_SAMPLES + center_stop
        x = samples[start:stop]
        if len(x) != win_n:
            break
        x = x - np.mean(x)
        p0 = np.dot(x, c0) ** 2 + np.dot(x, s0) ** 2
        p1 = np.dot(x, c1) ** 2 + np.dot(x, s1) ** 2
        bits.append(1 if p1 > p0 else 0)
        confidences.append(abs(p1 - p0) / max(p0 + p1, 1.0))
    return bits, confidences

def best_alternating_run(bits, confidences):
    best = {'score': -1.0, 'bit_index': 0, 'length': 0, 'polarity': 0}
    for polarity in (0, 1):
        run_start = 0
        run_conf = []
        for i, bit in enumerate(bits):
            expected = (i + polarity) & 1
            if bit == expected:
                if not run_conf:
                    run_start = i
                run_conf.append(confidences[i] if i < len(confidences) else 0.0)
            else:
                if len(run_conf) >= MIN_PREAMBLE_BITS:
                    score = len(run_conf) + float(np.mean(run_conf))
                    if score > best['score']:
                        best = {'score': score, 'bit_index': run_start, 'length': len(run_conf), 'polarity': polarity}
                run_conf = []
        if len(run_conf) >= MIN_PREAMBLE_BITS:
            score = len(run_conf) + float(np.mean(run_conf))
            if score > best['score']:
                best = {'score': score, 'bit_index': run_start, 'length': len(run_conf), 'polarity': polarity}
    return best

def search_frame(samples):
    best_alt = {'score': -1.0, 'offset': 0, 'bit_phase': 0, 'bit_index': 0, 'length': 0, 'bits': []}
    for offset in range(0, BIT_SAMPLES, OFFSET_STRIDE):
        bits, confidences = demod_bits(samples, offset)
        if len(bits) < 64:
            continue
        alt = best_alternating_run(bits, confidences)
        if alt['score'] > best_alt['score']:
            best_alt = {'score': alt['score'], 'offset': offset, 'bit_phase': alt['bit_index'] % 8, 'bit_index': alt['bit_index'], 'length': alt['length'], 'bits': bits}
        for phase in range(8):
            byte_values = bits_to_bytes_lsb(bits, phase)
            frame = find_frame(byte_values)
            if frame is not None:
                return {'frame': frame, 'offset': offset, 'bit_phase': phase, 'bytes': byte_values, 'bits': bits, 'preamble_score': alt['score'], 'preamble_bit': alt['bit_index'], 'preamble_len': alt['length']}
    return {'frame': None, 'offset': best_alt['offset'], 'bit_phase': best_alt['bit_phase'], 'bits': best_alt['bits'], 'preamble_score': best_alt['score'], 'preamble_bit': best_alt['bit_index'], 'preamble_len': best_alt['length']}

def payload_text(payload):
    return ''.join(chr(b) if 32 <= b <= 126 else '.' for b in payload)

print('Overlay loaded:', ctrl_name, dma_name)
print('FS_HZ = %.1f Hz, BIT_SAMPLES = %d, capture = %.3f s' % (FS_HZ, BIT_SAMPLES, SAMPLE_COUNT / FS_HZ))

In [ ]:
# Capture one segment and display 4 cycles of 1200 Hz.
raw, ch0, ch1 = capture_once()
samples = ch0 if CHANNEL == 'ch0' else ch1
samples = samples - np.mean(samples)

window_n = int(round(FS_HZ * 4 / SPACE_HZ))
search_n = min(len(samples), int(round(FS_HZ * 0.5)))
best_start = 0
best_vpp = -1
for start in range(0, max(1, search_n - window_n), max(1, window_n // 4)):
    seg = samples[start:start + window_n]
    vpp = float(np.max(seg) - np.min(seg))
    if vpp > best_vpp:
        best_vpp = vpp
        best_start = start

seg = samples[best_start:best_start + window_n]
t_ms = np.arange(len(seg)) / FS_HZ * 1000.0
p1200 = tone_power(seg, SPACE_HZ)
p2200 = tone_power(seg, MARK_HZ)

print('channel      =', CHANNEL)
print('window start = %d samples, %.3f ms' % (best_start, best_start / FS_HZ * 1000.0))
print('window len   = %d samples, %.3f ms' % (window_n, window_n / FS_HZ * 1000.0))
print('vpp          = %.1f ADC counts' % best_vpp)
print('E1200/E2200  = %.3e / %.3e' % (p1200, p2200))

plt.figure(figsize=(10, 4))
plt.plot(t_ms, seg, marker='o', markersize=3)
plt.grid(True)
plt.xlabel('Time (ms)')
plt.ylabel('ADC counts, DC removed')
plt.title('%s 1200 Hz check, 4 cycles' % CHANNEL.upper())
plt.show()

In [ ]:
# Repeatedly capture until one preamble/frame is detected, then print result and exit this cell.
MAX_ATTEMPTS = 60

for attempt in range(1, MAX_ATTEMPTS + 1):
    t0 = time()
    raw, ch0, ch1 = capture_once()
    samples = ch0 if CHANNEL == 'ch0' else ch1
    result = search_frame(samples)
    elapsed = time() - t0
    frame = result['frame']
    preamble_ms = (result['offset'] + result['preamble_bit'] * BIT_SAMPLES) / FS_HZ * 1000.0
    print('attempt %02d: %.3f s, preamble_score=%.3f, offset=%d, phase=%d, preamble=%.1f ms, len=%d bits' % (
        attempt, elapsed, result['preamble_score'], result['offset'], result['bit_phase'], preamble_ms, result['preamble_len']))

    if frame is None:
        continue

    print('\nAFSK SMS frame detected')
    print('addr        = 0x%02X' % frame['addr'])
    print('addr_ok     =', frame['addr_ok'])
    print('length      =', frame['length'])
    print('payload     =', payload_text(frame['payload']))
    print('payload_hex =', ' '.join('%02X' % b for b in frame['payload']))
    print('crc rx/calc = 0x%02X / 0x%02X' % (frame['rx_crc'], frame['calc_crc']))
    print('crc_ok      =', frame['crc_ok'])
    break
else:
    print('No complete frame detected after %d attempts.' % MAX_ATTEMPTS)